<h1>Semi-Supervised Learning & Label Propagation</h1>

In semi-supervised learning only **some** of the samples have labels, and the
algorithm estimates the labels of the rest using the labelled samples *and* the
shape of the unlabelled data.

**Label Propagation** is the older sibling of the Label Spreading algorithm from the
previous notebook. Both push known labels through a graph of similar points, with one
key difference: Label Propagation **clamps** the labelled samples — a flower that came
with a label keeps it, no matter what — while Label Spreading allows a little
flexibility in case some given labels are noisy.

This notebook also uses a different way of building the graph: a **knn kernel**,
where each flower is connected to its 5 nearest neighbours, and labels flow only
along those connections.

We use the same IRIS setup as the previous notebook — same seed, same 70% of labels
hidden — so the two algorithms can be compared directly at the end.

<h1>Step 1 : Import the libraries</h1>

In [ ]:
# import the libraries we need:
# numpy             -> random numbers and array handling
# datasets          -> sklearn's built-in datasets (includes IRIS)
# LabelPropagation  -> the semi-supervised algorithm for this lab
# metrics           -> honest marking: accuracy, per-class report, confusion matrix
import numpy as np
from sklearn import datasets
from sklearn.semi_supervised import LabelPropagation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


<h1>Step 2 : Import Dataset</h1>
IRIS ships inside scikit-learn, so there is no file to download — this always works.

In [ ]:
# load the IRIS dataset
iris = datasets.load_iris()

# quick orientation: how many flowers, how many measurements, which species
print("data shape:", iris.data.shape)          # (150 flowers, 4 measurements)
print("species:", list(iris.target_names))     # 0=setosa, 1=versicolor, 2=virginica

<h1>Step 3 : Hiding the labels</h1>
Same recipe as the previous notebook: a seeded random number generator hides about
70% of the labels by setting them to **-1** — scikit-learn's marker for "unlabelled".
Because the seed is the same, exactly the same 111 flowers lose their labels as in
the Label Spreading notebook, which makes the two algorithms directly comparable.

In [ ]:
# a random number generator with a fixed seed — same "random" choices every run
rand = np.random.RandomState(0)

# draw a random number 0-1 for each flower; True (about 70% of them) = hide its label
random_unlabeled_samples = rand.rand(len(iris.target)) <= 0.7

# copy the true labels, then overwrite the hidden ones with -1
# (we keep iris.target untouched — we need the truth later for marking)
un_labelled = np.copy(iris.target)
un_labelled[random_unlabeled_samples] = -1

# how did the split land?
print("total flowers:", len(un_labelled))
print("labels hidden:", random_unlabeled_samples.sum())
print("labels kept:  ", (~random_unlabeled_samples).sum())

<h1>Step 3.1: Inspect the labels</h1>
Every -1 below is a flower the model must label for itself.

In [ ]:
# print the label array the model will be given
print(un_labelled)

<h1>Step 4 : Fit the model and mark it honestly</h1>
We fit Label Propagation with a knn kernel: each flower connects to its 5 nearest
neighbours, and labels flow along those connections until every flower has one.

To mark it, we predict a species for the 111 **hidden** flowers and compare against
the true labels we kept aside — the per-class report shows where it does well and
where it struggles.

In [ ]:
# build the model: knn kernel, labels flow to each flower's 5 nearest neighbours
model = LabelPropagation(kernel='knn', n_neighbors=5)

# fit: the model sees measurements for all 150 flowers, but labels for only 39
model.fit(iris.data, un_labelled)

# predict a species for the hidden flowers only
predicted_labels = model.predict(iris.data[random_unlabeled_samples])
true_labels = iris.target[random_unlabeled_samples]

# per-class marking against the truth
print(classification_report(true_labels, predicted_labels,
                            target_names=['setosa', 'versicolor', 'virginica']))

# confusion matrix (rows = actual species, columns = predicted)
print(confusion_matrix(true_labels, predicted_labels))

# overall accuracy on the hidden flowers
print(f"\nAccuracy on the {random_unlabeled_samples.sum()} hidden flowers: "
      f"{accuracy_score(true_labels, predicted_labels):.1%}")


<h1>What did we learn?</h1>

- **Label Propagation recovered 102 of the 111 hidden labels (91.9%).** Setosa was
  perfect — it sits well apart from the other species. Nearly all the mistakes were
  virginica flowers called versicolor (8 of them): those two species genuinely
  overlap, so a flower's nearest neighbours can belong to the wrong side.
- **The knn kernel changes how labels travel.** Labels flow only between each flower
  and its 5 nearest neighbours — a stricter, more local rule than the previous
  notebook's approach.
- **Propagation vs Spreading, head to head:** on exactly the same 111 hidden flowers,
  Label Spreading recovered 103 and Label Propagation 102 — a one-flower difference.
  On clean data the two are near-twins; Spreading's flexibility matters more when
  some of the given labels are wrong.
- **The habit that matters:** once again we marked the model against the true labels
  we kept aside — never against its own predictions.